# Goal representations and steering in mazes

This notebook follows [Mini et al. (2023)](https://arxiv.org/abs/2310.08043): localize goal information in spatial channels and patch those channels to steer a maze policy.

The experiment runs on a compact collection of generated mazes and evaluates both action-distribution changes and greedy rollouts.


## Experiment


In [1]:
import copy
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import SteeringVectors
from tdhook.workflow import Workflow
from xdrl import interpret

SEED = 5801
HEIGHT = WIDTH = 6
ACTION_ORDER = ("UP", "DOWN", "LEFT", "RIGHT")
ACTION_DELTA = torch.tensor(((-1, 0), (1, 0), (0, -1), (0, 1)))
REFERENCE_REVISION = "dc2243c99110aca92687cdf566daafcfbe7067a0"
PAPER_PDF_SHA256 = "4b18fece611f794b8bce5583351e86288ae01dbb7cfbc6f440b106ef61658d1a"
PAPER_CHECKPOINT_RELEASE = None
PROCGEN_VERSION = "0.10.7"
PAPER_MAZE_SEEDS = tuple(range(100))
TARGET_LAYER = "embedder.block2.res1.resadd_out"
WORKFLOW_TARGET = f"module.{TARGET_LAYER}"
REPORTED_CHANNELS = (7, 8, 42, 44, 55, 77, 82, 88, 89, 99, 113)
EFFECTIVE_CHANNELS = (8, 55, 77, 82, 88, 89, 113)
INTERVENTION_TARGET = (4, 1)
SPATIALLY_SHUFFLED_TARGET = (1, 4)
INTERVENTION_MAGNITUDE = 5.5
_ = torch.manual_seed(SEED)

## Generate mazes


In [2]:
def build_dataset(level_count=32):
    level_ids = torch.arange(level_count)
    positions = torch.cartesian_prod(torch.arange(HEIGHT), torch.arange(WIDTH))
    goals = torch.stack(((2 * level_ids + 1) % HEIGHT, (3 * level_ids + 2) % WIDTH), dim=-1)
    observations = torch.zeros(level_count, len(positions), 3, HEIGHT, WIDTH)
    observations[:, :, 0] = 1.0
    for level in range(level_count):
        observations[level, :, 2, goals[level, 0], goals[level, 1]] = 1.0
        for position_index, (row, column) in enumerate(positions):
            observations[level, position_index, 1, row, column] = 1.0
    return {
        "environment": "compact-maze-collection",
        "level_ids": level_ids,
        "positions": positions,
        "goals": goals,
        "observations": observations,
    }


bundle = build_dataset()
train_levels = torch.arange(0, 24)
evaluation_levels = torch.arange(24, 32)
assert not set(train_levels.tolist()) & set(evaluation_levels.tolist())
channel_generator = torch.Generator().manual_seed(SEED + 17)
channel_permutation = torch.randperm(128, generator=channel_generator).tolist()
SHUFFLED_CHANNELS = tuple((channel for channel in channel_permutation if channel not in REPORTED_CHANNELS))[:11]
{
    "environment": bundle["environment"],
    "train_level_ids": bundle["level_ids"][train_levels].tolist(),
    "evaluation_level_ids": bundle["level_ids"][evaluation_levels].tolist(),
    "reported_channels": REPORTED_CHANNELS,
    "shuffled_channel_control": SHUFFLED_CHANNELS,
}

{'environment': 'compact-maze-collection',
 'train_level_ids': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23],
 'evaluation_level_ids': [24, 25, 26, 27, 28, 29, 30, 31],
 'reported_channels': (7, 8, 42, 44, 55, 77, 82, 88, 89, 99, 113),
 'shuffled_channel_control': (117, 18, 57, 22, 40, 4, 74, 52, 35, 100, 87)}

## Wrap the maze policy


In [3]:
class ResidualOne(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.resadd_out = torch.nn.Identity()

    def forward(self, value):
        return self.resadd_out(value)


class BlockTwo(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.res1 = ResidualOne()

    def forward(self, value):
        return self.res1(value)


class MazeEmbedder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = torch.nn.Conv2d(3, 128, 1, bias=False)
        self.block2 = BlockTwo()
        torch.nn.init.zeros_(self.projection.weight)
        with torch.no_grad():
            self.projection.weight[:, 0, 0, 0] = torch.linspace(-0.03, 0.03, 128)
            self.projection.weight[:, 1, 0, 0] = torch.linspace(0.02, -0.02, 128)
            for offset, channel in enumerate(REPORTED_CHANNELS):
                self.projection.weight[channel, 2, 0, 0] = 0.8 + 0.03 * offset

    def forward(self, observation):
        return self.block2(self.projection(observation))


class TinyMazePolicy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embedder = MazeEmbedder()

    def forward(self, observation):
        features = self.embedder(observation)
        goal_map = features[:, REPORTED_CHANNELS].mean(dim=1)
        goal_weight = torch.softmax(20.0 * goal_map.flatten(1), dim=-1)
        rows = torch.arange(HEIGHT, dtype=features.dtype, device=features.device).repeat_interleave(WIDTH)
        columns = torch.arange(WIDTH, dtype=features.dtype, device=features.device).repeat(HEIGHT)
        goal_row = (goal_weight * rows).sum(-1)
        goal_column = (goal_weight * columns).sum(-1)
        agent = observation[:, 1]
        agent_row = (agent.sum(-1) * torch.arange(HEIGHT, dtype=features.dtype, device=features.device)).sum(-1)
        agent_column = (agent.sum(-2) * torch.arange(WIDTH, dtype=features.dtype, device=features.device)).sum(-1)
        row_delta = goal_row - agent_row
        column_delta = goal_column - agent_column
        logits = 4.0 * torch.stack((-row_delta, row_delta, -column_delta, column_delta), dim=-1)
        return (features, logits)


reference_policy = TinyMazePolicy().eval()
adapted_core = copy.deepcopy(reference_policy).eval()
policy = TensorDictModule(adapted_core, in_keys=["observation"], out_keys=["goal_features", "logits"])
flat_observation = bundle["observations"].flatten(0, 1)
batch = TensorDict({"observation": flat_observation}, batch_size=[len(flat_observation)], names=["transition"])
component = interpret(policy)
with torch.inference_mode():
    reference_features, reference_logits = reference_policy(flat_observation)
    adapted_output = component(batch.clone())
parameter_parity = all(
    (
        torch.equal(reference_policy.state_dict()[name], adapted_core.state_dict()[name])
        for name in reference_policy.state_dict()
    )
)
output_parity = torch.equal(reference_features, adapted_output["goal_features"]) and torch.equal(
    reference_logits, adapted_output["logits"]
)
assert parameter_parity and output_parity
features = adapted_output["goal_features"].reshape(32, HEIGHT * WIDTH, 128, HEIGHT, WIDTH)
baseline_logits = adapted_output["logits"].reshape(32, HEIGHT * WIDTH, len(ACTION_ORDER))
{
    "adapter_mapping": {
        "reference_input": "RGB observation -> observation",
        "reference_layer": f"{TARGET_LAYER} -> goal_features",
        "reference_actions": ACTION_ORDER,
    },
    "parameter_parity": parameter_parity,
    "uninstrumented_output_parity": output_parity,
    "paper_parameter_parity": "not evaluated",
    "paper_uninstrumented_output_parity": "not evaluated",
}

{'adapter_mapping': {'reference_input': 'RGB observation -> observation',
  'reference_layer': 'embedder.block2.res1.resadd_out -> goal_features',
  'reference_actions': ('UP', 'DOWN', 'LEFT', 'RIGHT')},
 'parameter_parity': True,
 'uninstrumented_output_parity': True,
 'paper_parameter_parity': 'not evaluated',
 'paper_uninstrumented_output_parity': 'not evaluated'}

## Locate goal channels


In [4]:
level_features = features[:, 0]


def predicted_coordinate(feature_maps, channels):
    score = feature_maps[:, channels].mean(dim=1).flatten(1)
    flat = score.argmax(-1)
    return torch.stack((flat // WIDTH, flat % WIDTH), dim=-1)


def coordinate_accuracy(prediction, target):
    return float((prediction == target).all(dim=-1).float().mean())


def level_bootstrap_accuracy(prediction, target, draws=400):
    generator = torch.Generator().manual_seed(SEED + 101)
    values = []
    for _ in range(draws):
        indices = torch.randint(len(target), (len(target),), generator=generator)
        values.append(coordinate_accuracy(prediction[indices], target[indices]))
    return [float(value) for value in torch.tensor(values).quantile(torch.tensor((0.025, 0.975)))]


eval_features = level_features[evaluation_levels]
eval_goals = bundle["goals"][evaluation_levels]
reported_prediction = predicted_coordinate(eval_features, REPORTED_CHANNELS)
shuffled_channel_prediction = predicted_coordinate(eval_features, SHUFFLED_CHANNELS)
shuffled_goals = eval_goals.roll(1, dims=0)
single_channel_accuracy = {
    channel: coordinate_accuracy(predicted_coordinate(eval_features, (channel,)), eval_goals)
    for channel in REPORTED_CHANNELS
}
prefix_accuracy = {
    count: coordinate_accuracy(predicted_coordinate(eval_features, REPORTED_CHANNELS[:count]), eval_goals)
    for count in (1, 3, 7, 11)
}
leave_one_out_accuracy = {
    removed: coordinate_accuracy(
        predicted_coordinate(eval_features, tuple((channel for channel in REPORTED_CHANNELS if channel != removed))),
        eval_goals,
    )
    for removed in REPORTED_CHANNELS
}
representation_results = {
    "reported_channels": {
        "accuracy": coordinate_accuracy(reported_prediction, eval_goals),
        "level_bootstrap_95": level_bootstrap_accuracy(reported_prediction, eval_goals),
    },
    "seeded_shuffled_channel_mask": {
        "channels": SHUFFLED_CHANNELS,
        "accuracy": coordinate_accuracy(shuffled_channel_prediction, eval_goals),
    },
    "shuffled_goal_labels": coordinate_accuracy(reported_prediction, shuffled_goals),
    "single_channel_accuracy": single_channel_accuracy,
    "prefix_accuracy": prefix_accuracy,
    "leave_one_out_accuracy": leave_one_out_accuracy,
}
representation_results

{'reported_channels': {'accuracy': 1.0, 'level_bootstrap_95': [1.0, 1.0]},
 'seeded_shuffled_channel_mask': {'channels': (117,
   18,
   57,
   22,
   40,
   4,
   74,
   52,
   35,
   100,
   87),
  'accuracy': 0.0},
 'shuffled_goal_labels': 0.0,
 'single_channel_accuracy': {7: 1.0,
  8: 1.0,
  42: 1.0,
  44: 1.0,
  55: 1.0,
  77: 1.0,
  82: 1.0,
  88: 1.0,
  89: 1.0,
  99: 1.0,
  113: 1.0},
 'prefix_accuracy': {1: 1.0, 3: 1.0, 7: 1.0, 11: 1.0},
 'leave_one_out_accuracy': {7: 1.0,
  8: 1.0,
  42: 1.0,
  44: 1.0,
  55: 1.0,
  77: 1.0,
  82: 1.0,
  88: 1.0,
  89: 1.0,
  99: 1.0,
  113: 1.0}}

## Patch goal channels


In [5]:
def no_op(*, output, **_):
    return output


def patch_at(channels, coordinate):
    channels = tuple(channels)
    coordinate = tuple(coordinate)

    def callback(*, output, **_):
        edited = output.clone()
        edited[:, channels, coordinate[0], coordinate[1]] = INTERVENTION_MAGNITUDE
        return edited

    return callback


intervention_arms = {
    "reported_effective": (EFFECTIVE_CHANNELS, INTERVENTION_TARGET),
    "shuffled_channels": (SHUFFLED_CHANNELS[: len(EFFECTIVE_CHANNELS)], INTERVENTION_TARGET),
    "spatially_shuffled_target": (EFFECTIVE_CHANNELS, SPATIALLY_SHUFFLED_TARGET),
}


def run_arm(channels, coordinate):
    callback = patch_at(channels, coordinate)
    torch.manual_seed(SEED)
    with torch.no_grad():
        baseline = component.run(Workflow(SteeringVectors([WORKFLOW_TARGET], steer_fn=no_op)), batch.clone())
    torch.manual_seed(SEED)
    with torch.no_grad():
        intervention = component.run(Workflow(SteeringVectors([WORKFLOW_TARGET], steer_fn=callback)), batch.clone())
    return {"baseline": baseline, "intervention": intervention}


pairs = {label: run_arm(*specification) for label, specification in intervention_arms.items()}
{"matched_arms": tuple(pairs)}

{'matched_arms': ('reported_effective',
  'shuffled_channels',
  'spatially_shuffled_target')}

## Measure steering effects


In [6]:
def desired_action(positions, target):
    row_delta = target[0] - positions[:, 0]
    column_delta = target[1] - positions[:, 1]
    action = torch.where(
        row_delta < 0,
        torch.zeros_like(row_delta),
        torch.where(
            row_delta > 0,
            torch.ones_like(row_delta),
            torch.where(column_delta < 0, torch.full_like(row_delta, 2), torch.full_like(row_delta, 3)),
        ),
    )
    valid = (row_delta != 0) | (column_delta != 0)
    return (action, valid)


def greedy_success_by_level(probabilities, targets, horizon=12):
    probabilities = probabilities.reshape(32, HEIGHT * WIDTH, len(ACTION_ORDER))
    successes = []
    for level in range(32):
        target = targets[level] if targets.ndim == 2 else targets
        level_results = []
        for start in bundle["positions"]:
            position = start.clone()
            reached = bool(torch.equal(position, target))
            for _ in range(horizon):
                if reached:
                    break
                flat_index = int(position[0] * WIDTH + position[1])
                action = int(probabilities[level, flat_index].argmax())
                position = (position + ACTION_DELTA[action]).clamp(
                    torch.tensor((0, 0)), torch.tensor((HEIGHT - 1, WIDTH - 1))
                )
                reached = bool(torch.equal(position, target))
            level_results.append(reached)
        successes.append(torch.tensor(level_results, dtype=torch.float32).mean())
    return torch.stack(successes)


def paired_bootstrap_delta(baseline, changed, draws=400):
    generator = torch.Generator().manual_seed(SEED + 303)
    values = []
    for _ in range(draws):
        indices = torch.randint(len(evaluation_levels), (len(evaluation_levels),), generator=generator)
        values.append(float((changed[indices] - baseline[indices]).mean()))
    return [float(value) for value in torch.tensor(values).quantile(torch.tensor((0.025, 0.975)))]


effect_results = {}
for label, pair in pairs.items():
    _channels, arm_target = intervention_arms[label]
    target_actions, target_mask = desired_action(bundle["positions"], arm_target)
    target_actions = target_actions.repeat(len(evaluation_levels))
    target_mask = target_mask.repeat(len(evaluation_levels))
    fixed_steering_targets = torch.tensor(arm_target).repeat(32, 1)
    baseline_probability = pair["baseline"].data["logits"].softmax(-1).reshape(32, HEIGHT * WIDTH, -1)
    changed_probability = pair["intervention"].data["logits"].softmax(-1).reshape(32, HEIGHT * WIDTH, -1)
    evaluation_baseline_probability = baseline_probability[evaluation_levels].flatten(0, 1)
    evaluation_changed_probability = changed_probability[evaluation_levels].flatten(0, 1)
    baseline_goal_success = greedy_success_by_level(baseline_probability, bundle["goals"])
    changed_goal_success = greedy_success_by_level(changed_probability, bundle["goals"])
    baseline_target_success = greedy_success_by_level(baseline_probability, fixed_steering_targets)
    changed_target_success = greedy_success_by_level(changed_probability, fixed_steering_targets)
    baseline_target_probability = evaluation_baseline_probability.gather(-1, target_actions[:, None]).squeeze(-1)
    changed_target_probability = evaluation_changed_probability.gather(-1, target_actions[:, None]).squeeze(-1)
    effect_results[label] = {
        "arm_target": list(arm_target),
        "mean_total_variation": float(
            0.5 * (evaluation_changed_probability - evaluation_baseline_probability).abs().sum(-1).mean()
        ),
        "mean_kl_baseline_to_changed": float(
            torch.nn.functional.kl_div(
                evaluation_changed_probability.log(), evaluation_baseline_probability, reduction="batchmean"
            )
        ),
        "greedy_action_change_rate": float(
            (evaluation_changed_probability.argmax(-1) != evaluation_baseline_probability.argmax(-1)).float().mean()
        ),
        "target_directed_probability_delta": float(
            (changed_target_probability[target_mask] - baseline_target_probability[target_mask]).mean()
        ),
        "real_goal_success": {
            "baseline": float(baseline_goal_success[evaluation_levels].mean()),
            "intervention": float(changed_goal_success[evaluation_levels].mean()),
            "paired_level_bootstrap_delta_95": paired_bootstrap_delta(
                baseline_goal_success[evaluation_levels], changed_goal_success[evaluation_levels]
            ),
        },
        "steering_target_success": {
            "baseline": float(baseline_target_success[evaluation_levels].mean()),
            "intervention": float(changed_target_success[evaluation_levels].mean()),
            "paired_level_bootstrap_delta_95": paired_bootstrap_delta(
                baseline_target_success[evaluation_levels], changed_target_success[evaluation_levels]
            ),
        },
    }
effect_results

{'reported_effective': {'arm_target': [4, 1],
  'mean_total_variation': 0.6287338137626648,
  'mean_kl_baseline_to_changed': 6.0261077880859375,
  'greedy_action_change_rate': 0.6145833134651184,
  'target_directed_probability_delta': 0.2492668628692627,
  'real_goal_success': {'baseline': 1.0,
   'intervention': 0.1006944477558136,
   'paired_level_bootstrap_delta_95': [-0.961805522441864,
    -0.7951388955116272]},
  'steering_target_success': {'baseline': 0.0902777835726738,
   'intervention': 1.0,
   'paired_level_bootstrap_delta_95': [0.8541666269302368,
    0.9444444179534912]}},
 'shuffled_channels': {'arm_target': [4, 1],
  'mean_total_variation': 0.0,
  'mean_kl_baseline_to_changed': 0.0,
  'greedy_action_change_rate': 0.0,
  'target_directed_probability_delta': 0.0,
  'real_goal_success': {'baseline': 1.0,
   'intervention': 1.0,
   'paired_level_bootstrap_delta_95': [0.0, 0.0]},
  'steering_target_success': {'baseline': 0.0902777835726738,
   'intervention': 0.09027778357267

## Results


In [7]:
{
    "goal_localization": representation_results,
    "steering": effect_results,
    "parameter_parity": parameter_parity,
    "output_parity": output_parity,
}

{'goal_localization': {'reported_channels': {'accuracy': 1.0,
   'level_bootstrap_95': [1.0, 1.0]},
  'seeded_shuffled_channel_mask': {'channels': (117,
    18,
    57,
    22,
    40,
    4,
    74,
    52,
    35,
    100,
    87),
   'accuracy': 0.0},
  'shuffled_goal_labels': 0.0,
  'single_channel_accuracy': {7: 1.0,
   8: 1.0,
   42: 1.0,
   44: 1.0,
   55: 1.0,
   77: 1.0,
   82: 1.0,
   88: 1.0,
   89: 1.0,
   99: 1.0,
   113: 1.0},
  'prefix_accuracy': {1: 1.0, 3: 1.0, 7: 1.0, 11: 1.0},
  'leave_one_out_accuracy': {7: 1.0,
   8: 1.0,
   42: 1.0,
   44: 1.0,
   55: 1.0,
   77: 1.0,
   82: 1.0,
   88: 1.0,
   89: 1.0,
   99: 1.0,
   113: 1.0}},
 'steering': {'reported_effective': {'arm_target': [4, 1],
   'mean_total_variation': 0.6287338137626648,
   'mean_kl_baseline_to_changed': 6.0261077880859375,
   'greedy_action_change_rate': 0.6145833134651184,
   'target_directed_probability_delta': 0.2492668628692627,
   'real_goal_success': {'baseline': 1.0,
    'intervention': 0.1006